# 02 — Features: posting / profile text shaping for retrieval

**Project:** AI Career Copilot (H09)

We exercise `posting_text` and `profile_text` — the two text-shaping helpers — and the TF-IDF retriever's tokenisation regime. The goal is to confirm the retrieval primitive recovers semantically-related postings from a profile + query.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path.cwd().parent / 'src'))
from career_copilot.data import PROCESSED, load_all, make_training_artifacts
from career_copilot.features import (
    posting_text, profile_text, blend_query_with_profile, skill_set, required_skill_set,
)
from sklearn.feature_extraction.text import TfidfVectorizer

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110

In [ ]:
if (PROCESSED / 'postings.parquet').exists():
    postings, profiles = load_all()
else:
    postings, profiles = make_training_artifacts()

## 1. `posting_text` rendering

In [ ]:
for _, p in postings.head(2).iterrows():
    print(posting_text(p))
    print('---')

## 2. `profile_text` rendering

In [ ]:
for _, prof in profiles.head(2).iterrows():
    print(profile_text(prof.to_dict()))
    print('---')

## 3. Build the TF-IDF on the posting corpus

In [ ]:
vec = TfidfVectorizer(
    token_pattern=r'[A-Za-z][A-Za-z\-+#./ ]+', lowercase=True, ngram_range=(1, 2),
    max_features=4096, sublinear_tf=True,
)
docs = [posting_text(r) for _, r in postings.iterrows()]
X = vec.fit_transform(docs)
print('shape:', X.shape, 'vocab:', len(vec.vocabulary_))

## 4. Token mass distribution

In [ ]:
mass = np.asarray(X.sum(axis=0)).ravel()
fig, ax = plt.subplots(figsize=(8, 3.4))
ax.hist(mass, bins=40, color='#9c6644', edgecolor='white')
ax.set_xlabel('summed TF-IDF mass per token')
ax.set_title('Token mass distribution across postings')
plt.tight_layout(); plt.show()

## 5. Profile-blended query example

In [ ]:
prof = profiles.iloc[0].to_dict()
query = 'I want to move into a data role next year'
blended = blend_query_with_profile(query, prof)
print(blended)
from sklearn.metrics.pairwise import cosine_similarity
qv = vec.transform([blended])
sims = cosine_similarity(qv, X).ravel()
top = np.argsort(-sims)[:5]
postings.iloc[top][['job_id', 'title', 'dept', 'level']].assign(sim=sims[top].round(3))

## 6. Skill-set helpers — `skill_set` / `required_skill_set`

These produce sets used downstream by the gap-filling templates.

In [ ]:
have = skill_set(prof['skills'])
need = required_skill_set(postings.iloc[top[0]]['required_skills'])
print('have:', sorted(have))
print('need:', sorted(need))
print('missing:', sorted(need - have))

## 7. Hand-off

These shaping helpers feed `models.fit_retriever`, which persists the TF-IDF + sparse matrix to `models/rag_index.joblib`.